## 1. 라이브러리 및 환경 설정

In [3]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from lightgbm import LGBMRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

In [ ]:
%pip install xgboost

In [8]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import StackingRegressor
# 스태킹에 사용할 base 모델들 정의
base_models = [
    ('lgb', LGBMRegressor(n_estimators=1000, learning_rate=0.05, max_depth=7,
                          num_leaves=63, subsample=0.8, colsample_bytree=0.8,
                          reg_alpha=0.1, reg_lambda=0.1, random_state=42, verbose=-1)),
    ('rf', RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)),
    ('xgb', XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=7, 
                         subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0))
]

# 메타 모델 (스태킹 레벨 2)
meta_model = Ridge(alpha=1.0)
print(f"베이스 모델 개수: {len(base_models)}")
print(f"메타 모델: {meta_model.__class__.__name__}")

베이스 모델 개수: 3
메타 모델: Ridge


## 2. 데이터 로드

In [10]:
from sklearn.preprocessing import RobustScaler
train = pd.read_csv('./data/train.csv')
test = pd.read_csv('./data/test.csv')

print(f"학습 데이터 크기: {train.shape}")
print(f"테스트 데이터 크기: {test.shape}")

학습 데이터 크기: (250000, 94)
테스트 데이터 크기: (50000, 93)


## 3. 피처 및 타겟 설정

In [13]:
TARGET = 'avg_delay_minutes_next_30m'
ID_COLS = ['ID', 'layout_id', 'scenario_id']

feature_cols = [c for c in train.columns if c not in ID_COLS + [TARGET]]
print(f"피처 수: {len(feature_cols)}")

rs = RobustScaler()
train[feature_cols] = rs.fit_transform(train[feature_cols])
test[feature_cols] = rs.transform(test[feature_cols])


피처 수: 90


## 4. 모델 학습 (5-Fold CV)

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(train))
test_preds = np.zeros(len(test))

stacking_model = StackingRegressor(base_models,final_estimator=meta_model,cv=kf)

for fold, (tr_idx, val_idx) in enumerate(kf.split(train)):
    print(f"── Fold {fold + 1} ──")
    X_tr = train.loc[tr_idx, feature_cols]
    y_tr = train.loc[tr_idx, TARGET]
    X_val = train.loc[val_idx, feature_cols]
    y_val = train.loc[val_idx, TARGET]

    stacking_model.fit(X_tr, y_tr)

    oof_preds[val_idx] = stacking_model.predict(X_val)
    test_preds += stacking_model.predict(test[feature_cols]) / 5

── Fold 1 ──


## 5. 결과 확인

In [ ]:
oof_mae = mean_absolute_error(train[TARGET], oof_preds)
print(f"OOF MAE: {oof_mae:.4f}")

## 6. 제출 파일 생성

In [ ]:
submission = pd.DataFrame({'ID': test['ID'], TARGET: test_preds})
submission.to_csv('./submission.csv', index=False)
print("submission.csv 저장 완료.")